# 01. CDEC Full Natural Flow (FNF) Download & Zarr Consolidation

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/google-research/google-research/blob/master/flowracle/notebooks/data_preprocessing/01_download_cdec_fnf.ipynb)

This notebook programmatically downloads historical monthly (Sensor `65`) and
optional daily (Sensor `8`) unimpaired **Full Natural Flow (FNF)** records and
station metadata from the California Data Exchange Center (CDEC), consolidating
them into the Zarr store consumed by Flowracle:
- **Output path**: `${FLOWRACLE_DATA_DIR}/ca-monthly-fnf-extended/monthly_fnf_consolidated.zarr`
- **Watershed coverage (25 gauges)**: All 23 operational DWR Bulletin 120
  watersheds plus **OWL** (Owens River at Long Valley, Eastern Sierra) and
  **KRB** (Kern River at Bakersfield, Southern Sierra).

### Why Monthly FNF Is Used as Target & Lagged Predictor
Monthly FNF undergoes formal reservoir-storage, diversion, and evaporation
mass-balance verification by California DWR and partner water districts after
month-end accounting. In Flowracle, historical monthly FNF is therefore
treated as a **lagged predictor** restricted to lags $\le -1$ month prior to
the end-of-month predictor origin (i.e., month $m-2$ and earlier for a
forecast issued on the 1st of month $m$).

##### Copyright 2026 The Google Research Authors.

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

    http://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.

---
*Disclaimer: This is not an officially supported Google product.*

In [ ]:
!pip install -q "numpy>=1.26.0" "pandas>=2.0.0" "requests>=2.31.0" "beautifulsoup4>=4.12.0" "lxml>=4.9.0" "xarray>=2023.12.0" "zarr>=2.16.0,<3.0.0" "dask>=2023.12.0" "tqdm>=4.66.0"

In [ ]:
from datetime import datetime
import io
import os
import re
import shutil

from bs4 import BeautifulSoup
import numpy as np
import pandas as pd
import requests
import xarray as xr

FLOWRACLE_DATA_DIR = os.environ.get("FLOWRACLE_DATA_DIR", "./flowracle_data")
DATASET_DIR = os.path.join(FLOWRACLE_DATA_DIR, "ca-monthly-fnf-extended")
RAW_CSV_DIR = os.path.join(DATASET_DIR, "raw_csv")
os.makedirs(RAW_CSV_DIR, exist_ok=True)

# Set VERIFICATION_MODE = True for a fast 3-station smoke test.
VERIFICATION_MODE = False
FREQUENCY = (  # 'monthly' (Sensor 65, used by Flowracle) or 'daily' (Sensor 8)
    "monthly"
)
DEFAULT_START_DATE = "1900-01-01"
DEFAULT_END_DATE = datetime.now().strftime("%Y-%m-%d")

STATION_START_DATES = {
    "KWT": {"monthly": "1900-10-01", "daily": "1990-01-01"},
    "SDT": {"monthly": "1905-10-01", "daily": "1990-01-01"},
    "MSS": {"monthly": "1910-10-01", "daily": "1990-01-01"},
    "PSH": {"monthly": "1950-01-01", "daily": "1990-01-01"},
    "SIS": {"monthly": "1930-01-01", "daily": "1990-01-01"},
    "SBB": {"monthly": "1950-01-01", "daily": "2000-01-01"},
    "FTO": {"monthly": "1900-01-01", "daily": "1990-01-01"},
    "WWR": {"monthly": "1900-01-01", "daily": "1990-01-01"},
    "SBF": {"monthly": "1900-01-01", "daily": "1990-01-01"},
    "EFC": {"monthly": "1900-01-01", "daily": "1990-01-01"},
    "SNS": {"monthly": "1900-01-01", "daily": "2000-01-01"},
    "MKM": {"monthly": "1900-01-01", "daily": "1990-01-01"},
    "TRF": {"monthly": "1900-01-01", "daily": "1990-01-01"},
    "SCC": {"monthly": "1900-01-01", "daily": "1990-01-01"},
    "AMF": {"monthly": "1900-01-01", "daily": "1990-01-01"},
    "MRC": {"monthly": "1900-01-01", "daily": "1990-01-01"},
    "KRI": {"monthly": "1900-01-01", "daily": "1990-01-01"},
    "CSN": {"monthly": "1900-01-01", "daily": "2000-01-01"},
    "WFC": {"monthly": "1900-01-01", "daily": "1990-01-01"},
    "YRS": {"monthly": "1900-01-01", "daily": "1990-01-01"},
    "EWR": {"monthly": "1900-01-01", "daily": "1990-01-01"},
    "TLG": {"monthly": "1900-01-01", "daily": "1990-01-01"},
    "KGF": {"monthly": "1900-01-01", "daily": "1990-01-01"},
    "OWL": {"monthly": "1907-10-01"},
    "KRB": {"monthly": "1900-10-01"},
}

START_DATE = "2024-01-01" if VERIFICATION_MODE else DEFAULT_START_DATE
END_DATE = "2024-06-01" if VERIFICATION_MODE else DEFAULT_END_DATE

## 1. Station List & CDEC Metadata Retrieval

In [ ]:
FNF_25_STATIONS = [
    "AMF",
    "CSN",
    "EFC",
    "EWR",
    "FTO",
    "KGF",
    "KRB",
    "KRI",
    "KWT",
    "MKM",
    "MRC",
    "MSS",
    "OWL",
    "PSH",
    "SBB",
    "SBF",
    "SCC",
    "SDT",
    "SIS",
    "SNS",
    "TLG",
    "TRF",
    "WFC",
    "WWR",
    "YRS",
]


def fetch_station_metadata(stations, output_dir):
  """Fetches gauge latitude, longitude, elevation (ft), and basin metadata from CDEC."""
  metadata_list = []
  for station in stations:
    url = f"https://cdec.water.ca.gov/dynamicapp/staMeta?station_id={station}"
    try:
      response = requests.get(url, timeout=15)
      response.raise_for_status()
      soup = BeautifulSoup(response.content, "html.parser")
      meta = {"station_id": station}
      for td in soup.find_all("td"):
        text = td.get_text(strip=True)
        if text in [
            "Elevation",
            "County",
            "River Basin",
            "Hydrologic Area",
            "Operator",
        ]:
          next_td = td.find_next_sibling("td")
          if next_td:
            val = next_td.get_text(strip=True)
            if text == "Elevation":
              try:
                val = float(val.replace(",", "").split(" ")[0])
              except ValueError:
                pass
            meta[text.lower().replace(" ", "_")] = val

      for tr in soup.find_all("tr"):
        texts = [td.get_text(strip=True) for td in tr.find_all(["td", "th"])]
        if "Latitude" in texts and "Longitude" in texts:
          lat_idx = texts.index("Latitude")
          lon_idx = texts.index("Longitude")
          lat_match = re.search(r"([\d\.-]+)", texts[lat_idx + 1])
          lon_match = re.search(r"([\d\.-]+)", texts[lon_idx + 1])
          if lat_match:
            meta["latitude"] = float(lat_match.group(1))
          if lon_match:
            # CDEC lists West longitudes as positive or unsigned degrees on some pages
            lon_val = float(lon_match.group(1))
            meta["longitude"] = (
                -abs(lon_val) if abs(lon_val) <= 180.0 else lon_val
            )
      metadata_list.append(meta)
    except Exception as exc:
      print(f"Warning: failed metadata lookup for {station}: {exc}")

  df = pd.DataFrame(metadata_list).set_index("station_id")
  os.makedirs(output_dir, exist_ok=True)
  csv_path = os.path.join(output_dir, "station_metadata.csv")
  df.to_csv(csv_path)
  print(f"Saved station metadata for {len(df)} stations to {csv_path}")
  return df

## 2. Download Historical FNF CSV Records from CDEC

In [ ]:
def download_fnf(
    stations,
    start_date,
    end_date,
    frequency="monthly",
    start_dates_map=None,
    out_root=RAW_CSV_DIR,
):
  sensor, dur_code = (65, "M") if frequency == "monthly" else (8, "D")
  base_url = "https://cdec.water.ca.gov/dynamicapp/req/CSVDataServlet"
  freq_dir = os.path.join(out_root, frequency)
  os.makedirs(freq_dir, exist_ok=True)

  target_stations = stations[:3] if VERIFICATION_MODE else stations
  for station in target_stations:
    current_start = start_date
    if not VERIFICATION_MODE and start_dates_map and station in start_dates_map:
      if frequency in start_dates_map[station]:
        current_start = start_dates_map[station][frequency]
      else:
        continue
    params = {
        "Stations": station,
        "SensorNums": sensor,
        "dur_code": dur_code,
        "Start": current_start,
        "End": end_date,
    }
    try:
      resp = requests.get(base_url, params=params, timeout=30)
      resp.raise_for_status()
      df = pd.read_csv(io.StringIO(resp.text))
      if df.empty or "VALUE" not in df.columns:
        print(f"Skipping {station}: empty or invalid CSV response.")
        continue
      out_csv = os.path.join(freq_dir, f"{station}.csv")
      df.to_csv(out_csv, index=False)
      print(f"Saved {station:4s} -> {out_csv} ({len(df):,} rows)")
    except Exception as exc:
      print(f"Error downloading {station}: {exc}")


station_metadata_df = fetch_station_metadata(FNF_25_STATIONS, DATASET_DIR)
download_fnf(
    FNF_25_STATIONS,
    START_DATE,
    END_DATE,
    FREQUENCY,
    start_dates_map=STATION_START_DATES,
)

## 3. Consolidate into Zarr (`monthly_fnf_consolidated.zarr`)

In [ ]:
def consolidate_fnf_to_zarr(
    csv_dir, metadata_csv_path, zarr_out_path, frequency="monthly"
):
  datasets = []
  for filename in sorted(os.listdir(csv_dir)):
    if not filename.endswith(".csv"):
      continue
    station = os.path.splitext(filename)[0]
    df = pd.read_csv(
        os.path.join(csv_dir, filename),
        parse_dates=["DATE TIME", "OBS DATE"],
        low_memory=False,
    )
    if df.empty or "DATE TIME" not in df.columns or "VALUE" not in df.columns:
      continue
    cols_exist = [
        c
        for c in ["DATE TIME", "OBS DATE", "VALUE", "DATA_FLAG"]
        if c in df.columns
    ]
    df = df[cols_exist].copy()
    df["VALUE"] = pd.to_numeric(df["VALUE"], errors="coerce")
    rename_map = {
        "DATE TIME": "time",
        "OBS DATE": "obs_date",
        "VALUE": "fnf",
        "DATA_FLAG": "data_flag",
    }
    df = df.rename(columns=rename_map)
    df["station_id"] = station
    df = df[~df.duplicated(subset=["time", "station_id"], keep="last")]
    ds = df.set_index(["time", "station_id"]).to_xarray()
    ds["fnf"] = ds["fnf"].astype(np.float64)
    datasets.append(ds)

  combined = xr.concat(datasets, dim="station_id", join="outer").sortby(
      ["station_id", "time"]
  )
  if os.path.exists(metadata_csv_path):
    meta_df = pd.read_csv(metadata_csv_path, index_col="station_id")
    for col in meta_df.columns:
      vals = meta_df[col].reindex(combined["station_id"].values).values
      combined = combined.assign_coords({col: ("station_id", vals)})
    if "latitude" in combined.coords:
      combined.coords["latitude"].attrs["units"] = "degrees_north"
    if "longitude" in combined.coords:
      combined.coords["longitude"].attrs["units"] = "degrees_east"
    if "elevation" in combined.coords:
      combined.coords["elevation"].attrs["units"] = "feet"

  combined["fnf"].attrs[
      "description"
  ] = "Full Natural Flow (FNF) in Acre-Feet (unimpaired runoff)."
  combined.attrs.update({
      "units": "AF",
      "description": (
          f"Consolidated {frequency.capitalize()} Full Natural Flow (FNF)"
          " Dataset."
      ),
      "origin": "California Department of Water Resources (DWR) via CDEC",
      "sensor_number": 65 if frequency == "monthly" else 8,
  })
  if os.path.exists(zarr_out_path):
    shutil.rmtree(zarr_out_path)
  combined.to_zarr(zarr_out_path, mode="w")
  print(f"Wrote Zarr store: {zarr_out_path}")
  return combined


zarr_name = (
    "monthly_fnf_consolidated.zarr"
    if FREQUENCY == "monthly"
    else "daily_fnf_consolidated.zarr"
)
fnf_zarr_path = os.path.join(DATASET_DIR, zarr_name)
consolidated_ds = consolidate_fnf_to_zarr(
    os.path.join(RAW_CSV_DIR, FREQUENCY),
    os.path.join(DATASET_DIR, "station_metadata.csv"),
    fnf_zarr_path,
    frequency=FREQUENCY,
)
print(consolidated_ds)